# UTAU Auto OTO - Coupled ???+?? ?? (Colab)

? ???? **??? ?? ??? ?? ?????**? ?????.

?? ??:
- ???? ???
- ??? + MFA ??
- `prepare_pairs.py` ???
- `build_dataset.py` ?? CSV ??
- `build_mel_patch_cache.py` rawmel ?? ??
- `train.py` (`coupled_nn_v2_rawmel`) ??

?? ??:
- UI ??
- OTO ?? ??? ??/??
- ?? ?? ?? ?(?? ? ?? ??)



## 1) Drive ??? ? ?? ??

`LANGUAGE`, `FORMAT_TYPE`, Drive ??? ?? ?????.



In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

USE_DRIVE_REPO_SNAPSHOT = True
PROJECT_GIT_URL = 'https://github.com/SODAsoo07/Auto_OTO.git'
PROJECT_BRANCH = 'main'
DRIVE_REPO_SNAPSHOT = Path('/content/drive/MyDrive/UTAU_Auto_OTO_v3/Auto_OTO')

WORK_ROOT = Path('/content/utoa_colab')
PROJECT_ROOT = WORK_ROOT / 'Auto_OTO'
DATASET_ROOT = PROJECT_ROOT / 'dataset'

COPY_DATASET_FROM_DRIVE = True
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/UTAU_Auto_OTO_v3/dataset')

STAGE_FROM_SOURCE = False
COLAB_TRAINING_ROOTS_YAML = PROJECT_ROOT / 'ml' / 'configs' / 'training_data_roots.colab.yaml'
VOICEBANK_ROOTS = {
    'japanese': {'cv': [], 'vcv': [], 'cvvc': [], 'general': []},
    'korean': {'cv': [], 'cvc': [], 'cvvc': [], 'vcv': [], 'general': []},
}

LANGUAGE = 'korean'  # korean | japanese
FORMAT_TYPE = 'cvc'  # cv | cvc | cvvc | vcv | general

PREPARE_RESUME = True
PREPARE_RETRY_FAILED = True
PREPARE_LIMIT = 0
PREPARE_DRY_RUN = False
PREPARE_WORKERS = 0
AUTO_OTO_POLICY = 'generate-temp'  # require | generate-temp | generate-persist

RUNTIME_DEVICE = 'cuda'  # cuda | cpu | auto
EPOCHS = 70
BATCH_SIZE = 192
LEARNING_RATE = 1e-3
MIN_CONFIDENCE = 0.55

ARTIFACT_ROOT = WORK_ROOT / 'artifacts'
DATASET_CSV = ARTIFACT_ROOT / 'datasets' / LANGUAGE / f'dataset_{LANGUAGE}_{FORMAT_TYPE}_coupled.csv'
MEL_PATCH_CACHE_ROOT = ARTIFACT_ROOT / 'mel_patch_cache'
MODEL_OUT_DIR = ARTIFACT_ROOT / 'models' / LANGUAGE / FORMAT_TYPE / 'coupled_v2_rawmel'

COPY_ARTIFACTS_TO_DRIVE = True
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/UTAU_Auto_OTO_v3/colab_artifacts/coupled')

for p in [WORK_ROOT, ARTIFACT_ROOT, DATASET_CSV.parent, MEL_PATCH_CACHE_ROOT, MODEL_OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f'WORK_ROOT={WORK_ROOT}')
print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'DATASET_ROOT={DATASET_ROOT}')
print(f'DATASET_CSV={DATASET_CSV}')
print(f'MODEL_OUT_DIR={MODEL_OUT_DIR}')



## 2) ?? ??



In [ ]:
import json
import shutil
import subprocess

def run_cmd(args, env=None, cwd=None, capture_output=False):
    args = [str(a) for a in args]
    print('$', ' '.join(args))
    return subprocess.run(args, check=True, env=env, cwd=cwd, text=True, capture_output=capture_output)

def run_bash(cmd, env=None, cwd=None):
    print('$', cmd)
    return subprocess.run(cmd, shell=True, check=True, executable='/bin/bash', env=env, cwd=cwd, text=True)

def sync_project_snapshot():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    if USE_DRIVE_REPO_SNAPSHOT:
        if not DRIVE_REPO_SNAPSHOT.exists():
            raise FileNotFoundError(f'Drive snapshot not found: {DRIVE_REPO_SNAPSHOT}')
        PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        run_cmd(['rsync', '-a', '--delete', '--exclude', '.git', f'{DRIVE_REPO_SNAPSHOT}/', f'{PROJECT_ROOT}/'])
    else:
        run_cmd(['git', 'clone', '--depth', '1', '--branch', PROJECT_BRANCH, PROJECT_GIT_URL, str(PROJECT_ROOT)])



## 3) ???? ??? + ???/MFA ??



In [ ]:
import os

sync_project_snapshot()
os.chdir(PROJECT_ROOT)
print('cwd =', PROJECT_ROOT)

MFA_VERSION = '2.2.17'
MFA_PYTHON = '3.10'
MAMBA_ROOT = Path('/content/micromamba')
MICROMAMBA_EXE = Path('/content/bin/micromamba')
MFA_ENV_PREFIX = MAMBA_ROOT / 'envs' / 'mfa'
MFA_BIN = MFA_ENV_PREFIX / 'bin' / 'mfa'
MFA_PY_BIN = MFA_ENV_PREFIX / 'bin' / 'python'
MFA_ROOT_DIR = Path('/content/mfa_root')
MFA_ROOT_DIR.mkdir(parents=True, exist_ok=True)

run_cmd(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run_cmd([
    'python', '-m', 'pip', 'install', '-q',
    '-r', str(PROJECT_ROOT / 'requirements.txt'),
    '-r', str(PROJECT_ROOT / 'requirements-ml.txt'),
    'pyyaml',
])

if not MICROMAMBA_EXE.exists():
    Path('/content/bin').mkdir(parents=True, exist_ok=True)
    run_cmd(['wget', '-q', '-O', '/tmp/micromamba.tar.bz2', 'https://micro.mamba.pm/api/micromamba/linux-64/latest'])
    run_cmd(['tar', '-xjf', '/tmp/micromamba.tar.bz2', '-C', '/content', 'bin/micromamba'])

if not MFA_BIN.exists():
    run_cmd([
        str(MICROMAMBA_EXE), 'create', '-y',
        '-r', str(MAMBA_ROOT),
        '-p', str(MFA_ENV_PREFIX),
        '-c', 'conda-forge',
        f'python={MFA_PYTHON}',
        f'montreal-forced-aligner={MFA_VERSION}',
        'openfst=1.8.2',
        'kaldi=5.5.1068',
    ], env={**os.environ, 'MAMBA_ROOT_PREFIX': str(MAMBA_ROOT)})

run_cmd([
    str(MFA_PY_BIN), '-m', 'pip', 'install', '-q', '--upgrade',
    'pip', 'setuptools==80.9.0', 'wheel',
    'joblib>=1.3,<1.5',
    'python-mecab-ko', 'jamo', 'spacy', 'sudachipy', 'sudachidict-core',
])

os.environ['PATH'] = f"{MFA_ENV_PREFIX / 'bin'}:{os.environ['PATH']}"
os.environ['PYTHONUTF8'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['MFA_ROOT_DIR'] = str(MFA_ROOT_DIR)
os.environ['MPLBACKEND'] = 'Agg'
os.environ['UTOA_ML_PREPARE_MFA_PROFILE'] = 'default'
os.environ['UTOA_ML_MEL_PATCH_CACHE_DIR'] = str(MEL_PATCH_CACHE_ROOT)

mfa_env = os.environ.copy()
mfa_env['MPLBACKEND'] = 'Agg'

run_cmd([str(MFA_BIN), '--help'], env=mfa_env)
print('MFA_BIN =', MFA_BIN)



## 4) ???? ?? (Drive ?? ?? source stage)
- `STAGE_FROM_SOURCE=True`: ?? ????? ???? dataset ???
- `STAGE_FROM_SOURCE=False`: Drive? ?? dataset ??? ??



In [ ]:
import yaml

if STAGE_FROM_SOURCE:
    payload = {
        'japanese': VOICEBANK_ROOTS.get('japanese', {}),
        'korean': VOICEBANK_ROOTS.get('korean', {}),
        'notes': {
            'recursive_oto_search': True,
            'recursive_wav_search': True,
            'strip_pitch_suffix_for_matching': True,
        },
    }
    COLAB_TRAINING_ROOTS_YAML.parent.mkdir(parents=True, exist_ok=True)
    with open(COLAB_TRAINING_ROOTS_YAML, 'w', encoding='utf-8') as f:
        yaml.safe_dump(payload, f, allow_unicode=True, sort_keys=False)
    run_cmd([
        'python', '-X', 'utf8', str(PROJECT_ROOT / 'ml' / 'scripts' / 'coupled' / 'stage_sources.py'),
        '--config', str(COLAB_TRAINING_ROOTS_YAML),
        '--dataset-root', str(DATASET_ROOT),
    ], env=mfa_env, cwd=PROJECT_ROOT)
else:
    if COPY_DATASET_FROM_DRIVE:
        if not DRIVE_DATASET_ROOT.exists():
            raise FileNotFoundError(f'Drive dataset not found: {DRIVE_DATASET_ROOT}')
        DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
        run_cmd(['rsync', '-a', '--delete', f'{DRIVE_DATASET_ROOT}/', f'{DATASET_ROOT}/'])
    elif not DATASET_ROOT.exists():
        raise FileNotFoundError(f'dataset folder not found: {DATASET_ROOT}')

wav_count = sum(1 for _ in DATASET_ROOT.rglob('*.wav'))
oto_count = sum(1 for _ in DATASET_ROOT.rglob('oto.ini'))
print('wav_count =', wav_count)
print('oto_count =', oto_count)



## 5) prepare_pairs ?? (???)



In [ ]:
prepare_cmd = [
    'python', '-u', str(PROJECT_ROOT / 'ml' / 'scripts' / 'coupled' / 'prepare_pairs.py'),
    '--dataset-root', str(DATASET_ROOT),
]
if PREPARE_DRY_RUN:
    prepare_cmd.append('--dry-run')
if PREPARE_LIMIT and int(PREPARE_LIMIT) > 0:
    prepare_cmd += ['--limit', str(int(PREPARE_LIMIT))]
if PREPARE_RESUME:
    prepare_cmd.append('--resume')
if PREPARE_RETRY_FAILED:
    prepare_cmd.append('--retry-failed')
if PREPARE_WORKERS and int(PREPARE_WORKERS) > 0:
    prepare_cmd += ['--workers', str(int(PREPARE_WORKERS))]

run_cmd(prepare_cmd, env=mfa_env, cwd=PROJECT_ROOT)

report_path = DATASET_ROOT / '_manifest' / 'prepared_auto_pairs.json'
if not report_path.exists():
    raise FileNotFoundError(f'prepare report not found: {report_path}')
with open(report_path, 'r', encoding='utf-8') as f:
    report = json.load(f)
print(json.dumps(report.get('summary', {}), ensure_ascii=False, indent=2))



## 6) Coupled ?? CSV ?? (`build_dataset.py`)



In [ ]:
import pandas as pd

report_path = DATASET_ROOT / '_manifest' / 'prepared_auto_pairs.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
items = report.get('items', [])

lang_norm = LANGUAGE.strip().lower()
fmt_norm = FORMAT_TYPE.strip().lower()
prepared_status = {'prepared', 'prepared_existing'}

target_items = []
for item in items:
    if not isinstance(item, dict):
        continue
    if str(item.get('language', '')).strip().lower() != lang_norm:
        continue
    if str(item.get('format_type', '')).strip().lower() != fmt_norm:
        continue
    if str(item.get('status', '')).strip().lower() not in prepared_status:
        continue
    target_items.append(item)

if not target_items:
    raise RuntimeError(f'No prepared items for {LANGUAGE}/{FORMAT_TYPE}')

if DATASET_CSV.exists():
    DATASET_CSV.unlink()

build_script = PROJECT_ROOT / 'ml' / 'scripts' / 'coupled' / 'build_dataset.py'
for idx, item in enumerate(target_items):
    work_dir = str(item.get('work_dir', '')).strip()
    manual_oto = str(item.get('manual_oto', '')).strip()
    auto_oto = str(item.get('auto_oto', '')).strip()
    stage_root = str(item.get('stage_root', '')).strip()
    voicebank_id = Path(stage_root).name if stage_root else Path(work_dir).name

    if not work_dir or not manual_oto:
        continue

    cmd = [
        'python', '-X', 'utf8', str(build_script),
        '--lang', LANGUAGE,
        '--manual', manual_oto,
        '--tg-dir', work_dir,
        '--wav-dir', work_dir,
        '--out', str(DATASET_CSV),
        '--voicebank-id', voicebank_id,
        '--format-override', FORMAT_TYPE,
        '--auto-oto-policy', AUTO_OTO_POLICY,
    ]
    if auto_oto and Path(auto_oto).exists():
        cmd += ['--auto', auto_oto]
    if idx > 0:
        cmd.append('--append')

    run_cmd(cmd, env=mfa_env, cwd=PROJECT_ROOT)

if not DATASET_CSV.exists():
    raise FileNotFoundError(f'Dataset CSV not generated: {DATASET_CSV}')

df = pd.read_csv(DATASET_CSV)
print('dataset rows =', len(df))
print('columns =', len(df.columns))
print(df[['language', 'format_type', 'voicebank_id']].head(10))



## 7) rawmel ?? ?? + coupled v2(rawmel) ??



In [ ]:
cache_cmd = [
    'python', '-X', 'utf8', str(PROJECT_ROOT / 'ml' / 'scripts' / 'coupled' / 'build_mel_patch_cache.py'),
    '--dataset', str(DATASET_CSV),
    '--dataset-root', str(DATASET_ROOT),
]
cache_run = run_cmd(cache_cmd, env=mfa_env, cwd=PROJECT_ROOT, capture_output=True)
print(cache_run.stdout)
if cache_run.stderr:
    print(cache_run.stderr)

manifest_path = ''
for line in reversed((cache_run.stdout or '').splitlines()):
    line = line.strip()
    if line.endswith('manifest.json'):
        manifest_path = line
        break
if not manifest_path:
    candidates = sorted(MEL_PATCH_CACHE_ROOT.rglob('manifest.json'))
    if not candidates:
        raise RuntimeError('rawmel manifest.json not found')
    manifest_path = str(candidates[-1])

rawmel_cache_dir = str(Path(manifest_path).parent)
print('rawmel_cache_dir =', rawmel_cache_dir)

train_cmd = [
    'python', '-X', 'utf8', str(PROJECT_ROOT / 'ml' / 'scripts' / 'coupled' / 'train.py'),
    '--lang', LANGUAGE,
    '--format', FORMAT_TYPE,
    '--dataset', str(DATASET_CSV),
    '--out-dir', str(MODEL_OUT_DIR),
    '--backend', 'coupled_nn_v2_rawmel',
    '--rawmel-cache', rawmel_cache_dir,
    '--device', RUNTIME_DEVICE,
    '--epochs', str(int(EPOCHS)),
    '--batch-size', str(int(BATCH_SIZE)),
    '--learning-rate', str(float(LEARNING_RATE)),
    '--group-column', 'voicebank_id',
    '--min-confidence', str(float(MIN_CONFIDENCE)),
]
run_cmd(train_cmd, env=mfa_env, cwd=PROJECT_ROOT)

meta_path = MODEL_OUT_DIR / 'model_meta.json'
eval_path = MODEL_OUT_DIR / 'eval_summary.json'
model_path = MODEL_OUT_DIR / 'coupled_model.pt'

for p in [model_path, meta_path, eval_path]:
    print(p, 'exists=', p.exists())

if meta_path.exists():
    print('--- model_meta.json ---')
    print(meta_path.read_text(encoding='utf-8'))



## 8) ??? Drive ??



In [ ]:
if COPY_ARTIFACTS_TO_DRIVE:
    dst = DRIVE_ARTIFACT_ROOT / LANGUAGE / FORMAT_TYPE / 'coupled_v2_rawmel'
    dst.mkdir(parents=True, exist_ok=True)

    run_cmd(['rsync', '-a', f'{MODEL_OUT_DIR}/', f'{dst}/'])
    run_cmd(['rsync', '-a', str(DATASET_CSV), str(dst / DATASET_CSV.name)])

    print('copied_to =', dst)
else:
    print('COPY_ARTIFACTS_TO_DRIVE=False -> skip copy')

